# 연습 문제 1-10 풀이: 평균절대오차, 학습률

1-3절 연습 문제 1-10의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 방법으로도 구현할 수 있다.

- 본문 예제 코드는 `code_examples/ch01/` 아래 예제 노트북을 참고한다.

In [1]:
# torch 라이브러리 임포트와 시드 고정
import torch

SEED = 8
torch.manual_seed(SEED)
print(f'시드 고정: {SEED}')

시드 고정: 8


## 사전 준비

본문 1-3절의 데이터 생성, 모델, 손실 함수, 학습 루프([코드 1-28] ~ [코드 1-32])를 가져온다.
에포크마다 로그를 출력하면 실험 조합마다 수백 줄이 나오므로, 마지막 손실만 돌려주는 `train_quiet()`를 따로 구현해 사용한다.

In [2]:
SAMPLE_SIZE = 50    # 생성할 샘플의 수
LR = 0.0001         # 학습률
EPOCHS = 15         # 전체 학습 에포크
INIT = [0.96, -0.56]  # 파라미터 초깃값: 본문 [코드 1-33]과 같은 값

def generate_true_data(sample_size=SAMPLE_SIZE, true_a=4.9, true_b=0.0):
    X = torch.rand((sample_size, 1)) * 10                    # 관측시간
    Y_ideal = true_a * X ** 2 + true_b                       # 이상적인 낙하거리
    noise = torch.randn((sample_size, 1)) * Y_ideal * 0.1    # 관측 오차
    return X, Y_ideal + noise

X, Y_true = generate_true_data()
TRAIN_SIZE = int(SAMPLE_SIZE * 0.8)
X_train, Y_train = X[:TRAIN_SIZE], Y_true[:TRAIN_SIZE]
X_test, Y_test = X[TRAIN_SIZE:], Y_true[TRAIN_SIZE:]
print(f'훈련 데이터 {X_train.shape[0]}개, 평가 데이터 {X_test.shape[0]}개')

훈련 데이터 40개, 평가 데이터 10개


In [3]:
def model(parameters, X):
    return parameters[0] * X ** 2 + parameters[1]

def criterion(Y_pred, Y_true):
    return torch.mean((Y_pred - Y_true) ** 2)

def train_quiet(parameters, X, Y_true, epochs=EPOCHS, learning_rate=LR, criterion=criterion):
    """[코드 1-32]와 같은 학습 루프. 로그 대신 마지막 손실만 반환한다."""
    for _ in range(epochs):
        parameters.grad = None
        train_loss = criterion(model(parameters, X), Y_true)
        train_loss.backward()
        with torch.no_grad():
            parameters -= learning_rate * parameters.grad
    return train_loss.item()

## 연습 문제 1-10

> 모델 학습에 사용하는 손실 함수를 평균절대오차 손실 함수로 바꿔 모델을 학습하고, 손실 함수별 결과를 비교해 보자.
> 학습이 잘되지 않는다면 학습률을 조정해 보고, 두 손실 함수에 알맞은 학습률이 왜 서로 다른지도 생각해 보자.

**평균절대오차로 바꿔 학습하기**
- 평균절대오차: 오차의 절댓값의 평균
- 손실 함수만 바꾸고 학습률(0.0001)과 에포크(15)는 그대로
- 같은 조건의 평균제곱오차 결과와 비교

In [4]:
def criterion_mae(Y_pred, Y_true):
    return torch.mean(torch.abs(Y_pred - Y_true))       # 평균절대오차

for name, loss_fn in (('평균제곱오차', criterion), ('평균절대오차', criterion_mae)):
    parameters = torch.tensor(INIT, requires_grad=True)
    train_quiet(parameters, X_train, Y_train, criterion=loss_fn)
    print(f'{name}, 학습률 {LR}: a = {parameters[0].item():.4f}, b = {parameters[1].item():.4f}')

평균제곱오차, 학습률 0.0001: a = 5.1184, b = -0.4869
평균절대오차, 학습률 0.0001: a = 1.0032, b = -0.5585


**평균절대오차는 거의 학습되지 않는다: 기울기를 비교해 보자**
- 같은 지점에서 두 손실 함수의 기울기 크기를 찍어 본다
- 평균절대오차는 오차를 제곱하지 않아서 오차의 크기가 기울기에 반영되지 않는다

In [5]:
for name, loss_fn in (('평균제곱오차', criterion), ('평균절대오차', criterion_mae)):
    parameters = torch.tensor(INIT, requires_grad=True)
    loss = loss_fn(model(parameters, X_train), Y_train)
    loss.backward()
    print(f'{name}: 손실 {loss.item():10.2f}, a의 기울기 {parameters.grad[0].item():10.2f}')

평균제곱오차: 손실   29287.49, a의 기울기  -13866.57
평균절대오차: 손실     120.46, a의 기울기     -28.80


**학습률을 100배로 키우면**
- 평균절대오차의 기울기는 평균제곱오차의 기울기보다 훨씬 작다
- 학습률을 0.0001에서 0.01로 키워 다시 학습

In [6]:
LR_MAE = 0.01
parameters = torch.tensor(INIT, requires_grad=True)
train_quiet(parameters, X_train, Y_train, learning_rate=LR_MAE, criterion=criterion_mae)
print(f'평균절대오차, 학습률 {LR_MAE}: a = {parameters[0].item():.4f}, b = {parameters[1].item():.4f}')

평균절대오차, 학습률 0.01: a = 4.9227, b = -0.4210


**손실 함수마다 알맞은 학습률이 다르다**
- 학습률 0.0001, 0.001, 0.01, 0.1을 두 손실 함수에 모두 적용해 학습 후 a를 비교
- 값이 `inf`면 발산

In [7]:
print(f'{"학습률":>8} | {"평균제곱오차 a":>16} {"평균절대오차 a":>16}')
for lr in [0.0001, 0.001, 0.01, 0.1]:
    row = []
    for loss_fn in (criterion, criterion_mae):
        parameters = torch.tensor(INIT, requires_grad=True)
        train_quiet(parameters, X_train, Y_train, learning_rate=lr, criterion=loss_fn)
        row.append(f'{parameters[0].item():22.4g}')
    print(f'{lr:11} | {row[0]} {row[1]}')

     학습률 |         평균제곱오차 a         평균절대오차 a
     0.0001 |                  5.118                  1.003
      0.001 |              1.322e+06                  1.392
       0.01 |              1.785e+23                  4.923
        0.1 |                    inf                  3.848


**정리: 학습률은 손실 함수와 독립적인 값이 아니다**
- 평균제곱오차는 0.0001, 평균절대오차는 0.01 부근에서 학습이 잘 시작된다
- 평균제곱오차가 0.001면 발산하고, 평균절대오차가 0.1이면 a가 한곳에 안착하지 못한다
- 손실 함수를 바꾸면 학습률도 함께 조정해야 한다

**직접 해 보기: 학습률 0.1의 평균절대오차에서 에포크마다 a는 어떻게 변할까?**
- 에포크마다 a와 a의 기울기를 출력해 보자
- 먼저 어떤 값이 나올지 예측해 보자

In [8]:
parameters = torch.tensor(INIT, requires_grad=True)
for epoch in range(1, 7):
    parameters.grad = None
    loss = criterion_mae(model(parameters, X_train), Y_train)
    loss.backward()
    print(f'에포크 {epoch}: a = {parameters[0].item():.3f}, a의 기울기 = {parameters.grad[0].item():.2f}')
    with torch.no_grad():
        parameters -= 0.1 * parameters.grad
print(f'x**2의 평균: {(X_train ** 2).mean().item():.2f}')

에포크 1: a = 0.960, a의 기울기 = -28.80
에포크 2: a = 3.840, a의 기울기 = -28.80
에포크 3: a = 6.720, a의 기울기 = 28.76
에포크 4: a = 3.844, a의 기울기 = -28.80
에포크 5: a = 6.723, a의 기울기 = 28.76
에포크 6: a = 3.848, a의 기울기 = -28.80
x**2의 평균: 28.80


**확인: 평균절대오차의 기울기는 크기가 일정하다**
- 예측이 한쪽으로 치우쳐 있으면 기울기는 부호만 정해지고 크기는 `x**2`의 평균(28.80)으로 일정
- 한 에포크에 움직이는 거리는 학습률 * 28.8: 0.0001이면 0.003, 0.01이면 0.29, 0.1이면 2.88
- 0.1에서는 한 걸음이 너무 커서 a가 3.84와 6.72 사이를 왔다 갔다 한다